In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

OBJECTS_PATH = '../data/dust_generation/hourglass_objects.parquet'
PHOTOMETRY_PATH = '../data/dust_generation/hourglass_photometry.parquet'

SNR_MIN = 5.0
N_EPOCHS = 4
ZP_FLUXCAL = 27.5  # mag_calib = 27.5 - 2.5 * log10(fluxcal)
BATCH_SEED = None  # None draws a fresh random batch each run; set an int to reproduce one

BAND_ORDER = ['R', 'Z', 'Y', 'J', 'H', 'F']
BAND_COLORS = {
    band: plt.cm.turbo(position)
    for band, position in zip(BAND_ORDER, np.linspace(0.05, 0.95, len(BAND_ORDER)))
}

CLASS_ORDER = ['SN_Ia', 'CCSN', 'SN_Iax', 'SNIa-91bg', 'AGN', 'SLSN-I', 'TDE', 'ILOT', 'PISN', 'KN', 'Fixed_mag']

REDSHIFT_BINS = [0.0, 0.5, 1.0, 1.5, 2.0, np.inf]
REDSHIFT_LABELS = ['[0, 0.5)', '[0.5, 1)', '[1, 1.5)', '[1.5, 2)', '>= 2']

SURVEY_ORDER = ['WIDE', 'DEEP']
SURVEY_BANDS = {
    'WIDE': ['R', 'Z', 'Y', 'J'],
    'DEEP': ['Y', 'J', 'H', 'F'],
}


In [ ]:
phot = pq.read_table(
    PHOTOMETRY_PATH,
    columns=['cid', 'mjd', 'band', 'fluxcal', 'fluxcal_err', 'sim_mag_obs', 'zp'],
).to_pandas()

# 1. drop the simulation sentinel sim_mag_obs == 99 (no model SED at this phase — the source
#    physically does not exist there, so it is not a faint non-detection: it is dropped, not
#    turned into an upper limit)
phot = phot[phot['sim_mag_obs'] != 99.0]

# 2. token type: 'd' detection (SNR >= 5), 'u' upper limit (SNR < 5)
signal_to_noise = phot['fluxcal'] / phot['fluxcal_err']
phot['token_type'] = np.where(signal_to_noise >= SNR_MIN, 'd', 'u')

# 3. anchor each object at its first detection (Δt = 0) and keep its early window: the first
#    detection through its N_EPOCHS-th VISIT day. An epoch is a VISIT day (any band measured),
#    NOT a detection: the survey cadence is a uniform 5 days, so a faint source detected once
#    and then sitting below 5σ is still observed every 5 days — those later visits are
#    upper-limit tokens. Counting detection epochs instead would skip across them and inflate
#    Δt to ~100 days for flickering faint sources. Pre-detection visits are dropped (the first
#    detection is the anchor); objects with zero detections are dropped (never enter a
#    transient pipeline). Objects with < N_EPOCHS visit days keep their short window.
first_detection_mjd = phot[phot['token_type'] == 'd'].groupby('cid')['mjd'].min()
phot = phot[phot['mjd'] >= phot['cid'].map(first_detection_mjd)]
window_end_mjd = (
    phot.groupby('cid')['mjd']
    .apply(lambda mjd: np.sort(mjd.unique())[:N_EPOCHS][-1])
)
phot = phot[phot['mjd'] <= phot['cid'].map(window_end_mjd)]

# 4. token type 'n' (not observed): bands of the object's survey without a measurement on a
#    visit day, added with fluxcal / fluxcal_err / sim_mag_obs = NaN via the left merge
survey_per_object = pq.read_table(OBJECTS_PATH, columns=['cid', 'field']).to_pandas()
survey_per_object['survey'] = survey_per_object['field'].str.split('+').str[0]  # WIDE / DEEP (drop +PRISM)
survey_band_pairs = pd.DataFrame(
    [(survey, band) for survey, bands in SURVEY_BANDS.items() for band in bands],
    columns=['survey', 'band'],
)
expected_visits = (
    phot[['cid', 'mjd']]
    .drop_duplicates()
    .merge(survey_per_object[['cid', 'survey']], on='cid')
    .merge(survey_band_pairs, on='survey')
    .drop(columns='survey')
)
phot = expected_visits.merge(phot, on=['cid', 'mjd', 'band'], how='left')
phot['token_type'] = phot['token_type'].fillna('n')

# 5. observational-scenario channels for the matched KN twin (carried, NOT model inputs).
#    The KN dataloader injects a KN at this same visit/band/zp and rescales the error with
#    the source-Poisson differential (the sky+read+host background cancels in the difference):
#        sigma_KN^2 = fluxcal_err^2 + alpha * (F_KN - flux_true),   alpha = 10^((27.5 - zp)/2.5)
#    flux_true is the NOISELESS contaminant model flux from sim_mag_obs (SNANA identity), not
#    the noisy fluxcal. NaN on 'n' tokens (no measurement, so no scenario).
phot['flux_true'] = 10.0 ** ((ZP_FLUXCAL - phot['sim_mag_obs']) / 2.5)

print(f'Objects: {phot["cid"].nunique():,}   |   Rows: {len(phot):,}')
print(phot['token_type'].value_counts().to_string())
phot.head()


In [ ]:
def add_token_magnitudes(phot_table, zero_point=ZP_FLUXCAL, snr_limit=SNR_MIN):
    """Per-token apparent magnitude and its error, by token type.

    - 'd' detection: measured magnitude from fluxcal, sigma_mag from fluxcal_err.
    - 'u' upper limit: the per-visit 5σ limiting magnitude (the source is *fainter* than
      this), from fluxcal_err alone — never a fixed survey depth, since the depth actually
      reached is the information content of a non-detection. sigma_mag is undefined (NaN).
    - 'n' gap: magnitude channel masked entirely (NaN) — no information.

    The same 5σ-from-fluxcal_err recipe must be applied to injected KNe so the limit cannot
    become an injected-vs-native pipeline tell. Validated visually in the token light curves
    below (upper limits drawn at this 5σ magnitude).
    """
    phot_table = phot_table.copy()
    magnitude = np.full(len(phot_table), np.nan)
    sigma_magnitude = np.full(len(phot_table), np.nan)

    token_type = phot_table['token_type'].to_numpy()
    is_detection = token_type == 'd'
    is_upper_limit = token_type == 'u'
    flux = phot_table['fluxcal'].to_numpy()
    flux_error = phot_table['fluxcal_err'].to_numpy()

    magnitude[is_detection] = zero_point - 2.5 * np.log10(flux[is_detection])
    sigma_magnitude[is_detection] = (2.5 / np.log(10)) * flux_error[is_detection] / flux[is_detection]
    magnitude[is_upper_limit] = zero_point - 2.5 * np.log10(snr_limit * flux_error[is_upper_limit])

    phot_table['mag'] = magnitude
    phot_table['sigma_mag'] = sigma_magnitude
    return phot_table


phot = add_token_magnitudes(phot)
print(phot.groupby('token_type')['mag'].agg(['count', 'median']).to_string())
phot.head()


In [ ]:
EPOCHS_PER_WINDOW = 3  # the model sees at most 3 visit-day epochs; N_EPOCHS=4 keeps a shift buffer


def sample_epoch_window(object_table, shift_probability=0.20, n_epochs=EPOCHS_PER_WINDOW,
                        random_generator=None):
    """Select the (up to) n_epochs VISIT days the model sees for one object, anchored at the
    first detection.

    An epoch is a VISIT day (any band measured), not a detection. The cadence is a uniform
    5 days, so a faint source detected once and then below 5σ is still observed every 5 days
    — those later visits are upper-limit tokens within the window. Counting detection epochs
    instead would skip across them and blow Δt up to ~100 days for flickering faint sources.

    Default: the first n_epochs visit days from the first detection. With probability
    `shift_probability`, and only when shift-eligible (the 2nd visit day is itself a
    detection, so it can serve as the new Δt=0 anchor), slide the window forward by one epoch
    — drop epoch 1, start at epoch 2 — to simulate the ~20% of events caught already several
    days old. The draw is fresh each call (dataloader-time augmentation, not a fixed split).

    Faders with no detection at visit day 2 are never shift-eligible: they stay as the
    KN-like hard negatives, unshifted.

    Returns (window_rows, anchor_mjd, was_shifted): the d/u/n tokens of the selected visits,
    the anchor visit's mjd (for downstream Δt), and whether the shift fired.
    """
    if random_generator is None:
        random_generator = np.random.default_rng()

    visit_mjds = np.sort(object_table['mjd'].unique())
    detection_mjds = object_table.loc[object_table['token_type'] == 'd', 'mjd'].unique()

    is_shift_eligible = len(visit_mjds) >= 2 and (visit_mjds[1] in detection_mjds)
    was_shifted = is_shift_eligible and (random_generator.random() < shift_probability)

    start_index = 1 if was_shifted else 0
    window_mjds = visit_mjds[start_index:start_index + n_epochs]
    anchor_mjd = window_mjds[0]

    window_rows = object_table[object_table['mjd'].isin(window_mjds)].copy()
    return window_rows, anchor_mjd, was_shifted


In [ ]:
def class_by_redshift_table(cids, objects_path=OBJECTS_PATH, drop_classes=('AGN', 'Fixed_mag', 'KN'), plot=True):
    """Crosstab of class vs redshift bin over `cids`, restricted to contaminant transients.

    AGN and Fixed_mag are dropped (not transients); KN is dropped because it is the
    signal class, reported separately rather than in the contaminant census. Prints
    the table and its LaTeX version (paper-ready) and returns the DataFrame.
    """
    objects_df = pq.read_table(objects_path, columns=['cid', 'class', 'z_cmb']).to_pandas()
    objects_df = objects_df[objects_df['cid'].isin(cids)]
    objects_df = objects_df[~objects_df['class'].isin(drop_classes)]
    objects_df['z_bin'] = pd.cut(
        objects_df['z_cmb'],
        bins=REDSHIFT_BINS,
        labels=REDSHIFT_LABELS,
        right=False,
    )

    table = pd.crosstab(objects_df['class'], objects_df['z_bin'])
    table = table.reindex(
        index=[cls for cls in CLASS_ORDER if cls in table.index and cls not in drop_classes],
        columns=REDSHIFT_LABELS,
        fill_value=0,
    )
    table['total'] = table.sum(axis=1)
    table.loc['total'] = table.sum(axis=0)

    print(table.to_string())

    latex_column_map = {
        '[0, 0.5)':  r'$z < 0.5$',
        '[0.5, 1)':  r'$0.5$--$1$',
        '[1, 1.5)':  r'$1$--$1.5$',
        '[1.5, 2)':  r'$1.5$--$2$',
        '>= 2':      r'$z \geq 2$',
        'total':     r'\textbf{Total}',
    }
    latex_table_df = table.rename(columns=latex_column_map)
    latex_table_df.index.name = 'Class'

    latex_str = latex_table_df.to_latex(
        caption='Contaminant counts per spectroscopic class and CMB-frame redshift bin '
                '(transients surviving the SNR\\,+\\,4-epoch selection; '
                'AGN, Fixed\\textunderscore{}mag and KN excluded).',
        label='tab:class_redshift',
        column_format='l' + 'r' * (len(REDSHIFT_LABELS) + 1),
        bold_rows=False,
        escape=False,
    )
    # wrap tabular in \resizebox so the table fits a single journal column
    latex_str = latex_str.replace(
        r'\begin{tabular}',
        r'\resizebox{\columnwidth}{!}{\begin{tabular}',
    )
    latex_str = latex_str.replace(r'\end{tabular}', r'\end{tabular}}')
    print('\n' + latex_str)

    if plot:
        plot_class_redshift_heatmap(table.drop(index='total', columns='total'))

    return table, latex_str


def plot_class_redshift_heatmap(counts):
    figure, axis = plt.subplots(figsize=(7, 0.55 * len(counts) + 1.5))
    matrix = counts.to_numpy(dtype=float)
    image = axis.imshow(np.log10(matrix + 1), cmap='viridis', aspect='auto')
    axis.set_xticks(range(counts.shape[1]))
    axis.set_xticklabels(counts.columns, rotation=0)
    axis.set_yticks(range(counts.shape[0]))
    axis.set_yticklabels(counts.index)
    axis.set_xlabel('redshift bin')
    for row_index in range(counts.shape[0]):
        for column_index in range(counts.shape[1]):
            value = int(matrix[row_index, column_index])
            axis.text(column_index, row_index, f'{value:,}', ha='center', va='center',
                      color='white' if np.log10(value + 1) < np.log10(matrix + 1).max() * 0.6 else 'black',
                      fontsize=8)
    colorbar = figure.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    colorbar.set_label(r'$\log_{10}(N + 1)$')
    axis.set_title('Transient class vs redshift bin')
    plt.tight_layout()
    plt.savefig('hourglass_class_redshift_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()


# table over the transients that survive the SNR + 4-epoch cut (AGN, Fixed_mag excluded)
redshift_class_table, redshift_class_latex = class_by_redshift_table(phot['cid'].unique())


In [ ]:
def plot_sample_light_curves(
    objects_path=OBJECTS_PATH,
    photometry_path=PHOTOMETRY_PATH,
    snr_min=SNR_MIN,
    classes_to_show=None,
    objects_per_class=3,
    random_seed=42,
    plot=True,
):
    if classes_to_show is None:
        classes_to_show = ['SN_Ia', 'CCSN', 'SLSN-I', 'AGN', 'TDE']

    objects_df = pq.read_table(
        objects_path,
        columns=['cid', 'class', 'sub_class', 'field', 'z_cmb'],
    ).to_pandas()

    rng = np.random.default_rng(random_seed)
    selected_cids = {}
    for target_class in classes_to_show:
        candidates = objects_df[objects_df['class'] == target_class]['cid'].tolist()
        n_pick = min(objects_per_class, len(candidates))
        if n_pick == 0:
            continue
        picked = rng.choice(candidates, size=n_pick, replace=False).tolist()
        selected_cids[target_class] = picked

    all_cids = [cid for cids in selected_cids.values() for cid in cids]

    phot_df = pq.read_table(
        photometry_path,
        columns=['cid', 'mjd', 'band', 'fluxcal', 'fluxcal_err'],
        filters=[
            ('cid', 'in', all_cids),
            ('fluxcal', '>', 0.0),
            ('fluxcal_err', '>', 0.0),
        ],
    ).to_pandas()
    phot_df['snr'] = phot_df['fluxcal'] / phot_df['fluxcal_err']
    phot_df['mag_calib'] = ZP_FLUXCAL - 2.5 * np.log10(phot_df['fluxcal'])
    phot_df['sigma_mag'] = (2.5 / np.log(10)) * phot_df['fluxcal_err'] / phot_df['fluxcal']

    meta_df = objects_df[objects_df['cid'].isin(all_cids)].set_index('cid')

    if not plot:
        return phot_df

    number_of_rows = len(classes_to_show)
    number_of_columns = objects_per_class

    figure, axes = plt.subplots(
        number_of_rows, number_of_columns,
        figsize=(5.5 * number_of_columns, 3.5 * number_of_rows),
        squeeze=False,
    )

    for row_index, target_class in enumerate(classes_to_show):
        cids = selected_cids.get(target_class, [])
        for column_index in range(number_of_columns):
            axis = axes[row_index][column_index]
            if column_index >= len(cids):
                axis.set_visible(False)
                continue

            cid = cids[column_index]
            object_phot = phot_df[phot_df['cid'] == cid].copy()
            object_meta = meta_df.loc[cid]

            for band in BAND_ORDER:
                band_phot = object_phot[object_phot['band'] == band]
                detected = band_phot[band_phot['snr'] >= snr_min]
                non_detected = band_phot[band_phot['snr'] < snr_min]
                color = BAND_COLORS[band]

                if len(detected) > 0:
                    axis.errorbar(
                        detected['mjd'],
                        detected['mag_calib'],
                        yerr=detected['sigma_mag'],
                        fmt='o',
                        color=color,
                        markersize=4,
                        lw=1.0,
                        capsize=2,
                        label=band,
                        alpha=0.85,
                    )
                if len(non_detected) > 0:
                    axis.scatter(
                        non_detected['mjd'],
                        non_detected['mag_calib'],
                        marker='v',
                        color=color,
                        s=18,
                        alpha=0.25,
                    )

            axis.invert_yaxis()
            axis.set_xlabel('MJD', fontsize=9)
            axis.set_ylabel(r'mag$_{\rm calib}$ [mag]', fontsize=9)
            field_value = object_meta['field'] if isinstance(object_meta, pd.Series) else object_meta.iloc[0]['field']
            z_value = object_meta['z_cmb'] if isinstance(object_meta, pd.Series) else object_meta.iloc[0]['z_cmb']
            sub_value = object_meta['sub_class'] if isinstance(object_meta, pd.Series) else object_meta.iloc[0]['sub_class']
            axis.set_title(
                f'{target_class} / {sub_value}  |  cid={cid}\nfield={field_value}  z={z_value:.3f}',
                fontsize=8,
            )
            axis.grid(alpha=0.2)
            if row_index == 0 and column_index == number_of_columns - 1:
                handles = [
                    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=BAND_COLORS[b], markersize=7, label=b)
                    for b in BAND_ORDER
                ]
                axis.legend(handles=handles, frameon=False, fontsize=8, loc='upper right')

    plt.suptitle(
        f'Sample light curves — {objects_per_class} random objects per class  '
        f'(circles: SNR ≥ {snr_min}, triangles: SNR < {snr_min})',
        fontsize=12, y=1.01,
    )
    plt.tight_layout()
    plt.savefig('hourglass_sample_light_curves.png', dpi=150, bbox_inches='tight')
    plt.show()

    return phot_df


sample_phot_df = plot_sample_light_curves(
    classes_to_show=['SN_Ia', 'CCSN', 'SLSN-I', 'AGN', 'TDE'],
    objects_per_class=3,
    random_seed=BATCH_SEED,
    plot=True,
)


In [ ]:
def select_token_windows(phot_df, objects_path=OBJECTS_PATH, classes_to_show=None,
                         objects_per_class=3, shift_probability=0.20, random_seed=42):
    """Pick objects_per_class random objects per class and sample each one's 3-epoch window
    (possibly shifted) with sample_epoch_window. Returns one record per object — metadata +
    the sampled window rows + the shift flag — so the selection can be reviewed independently
    of the plot.
    """
    if classes_to_show is None:
        classes_to_show = ['SN_Ia', 'CCSN', 'SLSN-I', 'TDE']

    objects_df = pq.read_table(
        objects_path, columns=['cid', 'class', 'sub_class', 'field', 'z_cmb'],
    ).to_pandas()
    available_cids = set(phot_df['cid'].unique())
    meta_df = objects_df.set_index('cid')

    cid_generator = np.random.default_rng(random_seed)
    shift_generator = np.random.default_rng(random_seed)

    selection = []
    for target_class in classes_to_show:
        candidates = objects_df[
            (objects_df['class'] == target_class) & (objects_df['cid'].isin(available_cids))
        ]['cid'].tolist()
        n_pick = min(objects_per_class, len(candidates))
        if n_pick == 0:
            continue
        for cid in cid_generator.choice(candidates, size=n_pick, replace=False):
            window_rows, anchor_mjd, was_shifted = sample_epoch_window(
                phot_df[phot_df['cid'] == cid],
                shift_probability=shift_probability,
                random_generator=shift_generator,
            )
            meta = meta_df.loc[cid]
            selection.append({
                'target_class': target_class,
                'cid': int(cid),
                'sub_class': meta['sub_class'],
                'field': meta['field'],
                'z_cmb': float(meta['z_cmb']),
                'n_visits': int(window_rows['mjd'].nunique()),
                'anchor_mjd': anchor_mjd,
                'was_shifted': was_shifted,
                'window_rows': window_rows,
            })
    return selection


token_window_selection = select_token_windows(
    phot, classes_to_show=['SN_Ia', 'CCSN', 'SLSN-I', 'TDE'], objects_per_class=3,
    random_seed=BATCH_SEED,
)
pd.DataFrame(token_window_selection).drop(columns='window_rows')


In [ ]:
TOKEN_MARKERS = {'d': 'o', 'u': 'v', 'n': 's'}
TOKEN_LABELS = {'d': 'detection (SNR ≥ 5)', 'u': '5σ upper limit', 'n': 'not observed'}


def plot_token_light_curves(selection):
    """Draw the sampled token windows produced by select_token_windows — one panel per
    object, rows grouped by class. Circles = detection (mag ± σ), downward triangles = 5σ
    upper limit (source is fainter, no error bar), squares (foot strip) = not-observed band.
    The title flags the early {1,2,3} vs shifted {2,3,4} window.
    """
    records_by_class = {}
    for record in selection:
        records_by_class.setdefault(record['target_class'], []).append(record)

    classes_to_show = list(records_by_class.keys())
    number_of_rows = len(classes_to_show)
    number_of_columns = max(len(records) for records in records_by_class.values())

    figure, axes = plt.subplots(
        number_of_rows, number_of_columns,
        figsize=(5.5 * number_of_columns, 3.5 * number_of_rows),
        squeeze=False,
    )

    for row_index, target_class in enumerate(classes_to_show):
        records = records_by_class[target_class]
        for column_index in range(number_of_columns):
            axis = axes[row_index][column_index]
            if column_index >= len(records):
                axis.set_visible(False)
                continue

            record = records[column_index]
            window_rows = record['window_rows']

            # mag / sigma_mag already set per token type by add_token_magnitudes:
            #   'd' -> measured magnitude + error;  'u' -> per-visit 5σ limiting magnitude (σ = NaN)
            measured = window_rows[window_rows['token_type'].isin(['d', 'u'])]
            measured = measured[measured['mag'].notna()]

            faintest_mag = measured['mag'].max() if len(measured) > 0 else 26.0
            strip_mag = faintest_mag + 0.8

            for band in BAND_ORDER:
                color = BAND_COLORS[band]
                band_measured = measured[measured['band'] == band]

                detections = band_measured[band_measured['token_type'] == 'd']
                if len(detections) > 0:
                    axis.errorbar(
                        detections['mjd'],
                        detections['mag'],
                        yerr=detections['sigma_mag'],
                        fmt=TOKEN_MARKERS['d'],
                        color=color,
                        markersize=5,
                        markeredgecolor='black',
                        markeredgewidth=0.4,
                        lw=1.0,
                        capsize=2,
                        alpha=0.85,
                    )

                # upper limits at the 5σ limiting magnitude, no error bar — the triangle
                # points down: the source is fainter than this
                upper_limits = band_measured[band_measured['token_type'] == 'u']
                if len(upper_limits) > 0:
                    axis.scatter(
                        upper_limits['mjd'],
                        upper_limits['mag'],
                        marker=TOKEN_MARKERS['u'],
                        color=color,
                        s=45,
                        edgecolor='black',
                        linewidth=0.4,
                        alpha=0.5,
                    )

            # not-observed bands: one square per missing band on that visit, stacked
            # downward so the count of missing bands is readable at a glance
            square_spacing = 0.25
            not_observed = window_rows[window_rows['token_type'] == 'n']
            for visit_mjd, visit_group in not_observed.groupby('mjd'):
                missing_bands = [band for band in BAND_ORDER if band in visit_group['band'].values]
                for stack_index, band in enumerate(missing_bands):
                    axis.scatter(
                        visit_mjd,
                        strip_mag + stack_index * square_spacing,
                        marker=TOKEN_MARKERS['n'],
                        facecolor=BAND_COLORS[band],
                        edgecolor='black',
                        linewidth=0.4,
                        s=28,
                        alpha=0.7,
                    )

            axis.axhline(faintest_mag + 0.4, color='0.7', lw=0.8, ls='--')
            axis.invert_yaxis()
            axis.set_xlabel('MJD', fontsize=9)
            axis.set_ylabel(r'mag$_{\rm calib}$ [mag]', fontsize=9)
            shift_tag = 'shifted {2,3,4}' if record['was_shifted'] else 'early {1,2,3}'
            axis.set_title(
                f"{record['target_class']} / {record['sub_class']}  |  cid={record['cid']}  [{shift_tag}]\n"
                f"field={record['field']}  z={record['z_cmb']:.3f}",
                fontsize=8,
            )
            axis.grid(alpha=0.2)

    token_handles = [
        plt.Line2D([0], [0], marker=TOKEN_MARKERS[token_type], color='0.3', linestyle='none',
                   markersize=8, markeredgecolor='black', markeredgewidth=0.4, label=TOKEN_LABELS[token_type])
        for token_type in ['d', 'u', 'n']
    ]
    band_handles = [
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=BAND_COLORS[band],
                   markersize=7, label=band)
        for band in BAND_ORDER
    ]
    figure.legend(handles=token_handles + band_handles, loc='upper center',
                  ncol=len(token_handles) + len(band_handles), frameon=False, fontsize=9,
                  bbox_to_anchor=(0.5, 1.02))

    plt.suptitle('Sampled 3-epoch token windows — circle: detection, triangle: 5σ upper limit, '
                 'square (foot strip): not observed  (title flags early vs shifted window)',
                 fontsize=12, y=1.05)
    plt.tight_layout()
    plt.savefig('hourglass_token_light_curves.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_token_light_curves(token_window_selection)


## Light-curve definition and the explosion phase relative to the cadence

A light curve here is the per-object time series of band measurements, tokenised by epoch. With a Roman cadence of one visit every **5 days**, the model consumes the first `N_EPOCHS` epochs. When we later define the kilonova prefix this raises a question: **how old is the transient when its first epoch is observed?**

The explosion does not land on a visit. Between visit A and visit B there is a 5-day window, and the explosion falls at a **uniformly random** point inside it:

```
visit A                                            visit B
   |---------|---------|---------|---------|---------|
 day 0       1         2         3         4       day 5
   └── explodes here: B sees it at age 4–5 days  (the extreme case)
                                           └── explodes here: B sees it at age 0–1 day
```

Three consequences of the uniform phase:

1. If it explodes just after visit A (first day of the window), the first epoch that catches it (visit B) sees it **already 4–5 days old**.
2. If it explodes just before visit B (last day of the window), B catches it **fresh** (0–1 day).
3. Because the phase is uniform, the probability of landing in any sub-interval is just *(length of the sub-interval) / 5*.

So the **age at the first epoch is a continuous variable, uniform on [0, 5) days**, not a two-outcome coin. There is no single "fraction of delayed events" — it depends on where you set the threshold. The defensible statements:

- **On average the event reaches its first epoch at 2.5 days of age** — no arbitrary threshold, the most honest number.
- Equivalently, **50% of the time the event is older than 2.5 days (half an epoch)** at first detection.
- **In 1 of 5 events (20%) the first epoch lags by ≥ 4 days** — explosion in the first day of the window, 1/5 of the uniform phase. (A ≥ 3-day threshold would give 2/5 = 40%, etc.)

**Why this matters for the kilonova prefix:** since the phase is uniform with respect to the cadence, in roughly 1 of every 5 events the first visit occurs with ≥ 4 days of delay. To cover that regime we also evaluate the model on the **shifted prefix `{2, 3, 4}`** rather than only the early `{0, 1, 2}`, so performance is measured on the late-onset events the cadence will inevitably produce.


In [ ]:
def plot_epoch_sampling(detection_epochs_per_object, realized_shift_fraction,
                        shift_probability, window_token_counts):
    figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    counts = detection_epochs_per_object.value_counts().sort_index()
    bar_colors = ['crimson' if n == 1 else '0.5' for n in counts.index]
    axes[0].bar(counts.index.astype(str), counts.values, color=bar_colors)
    for x, value in zip(range(len(counts)), counts.values):
        axes[0].text(x, value, f'{value:,}', ha='center', va='bottom', fontsize=8)
    axes[0].set_xlabel('detection visits per object')
    axes[0].set_ylabel('objects')
    axes[0].set_title('red = single-epoch fast faders (hard negatives, never shifted)')

    axes[1].hist(window_token_counts, bins=range(1, 15), color='steelblue', alpha=0.8,
                 align='left', rwidth=0.85)
    axes[1].axvline(4 * EPOCHS_PER_WINDOW, color='0.3', ls='--', lw=1,
                    label=f'max = {4 * EPOCHS_PER_WINDOW} tokens')
    axes[1].set_xlabel('tokens in sampled window')
    axes[1].set_ylabel('objects (sampled)')
    axes[1].set_title(
        f'realized shift fraction (eligible) = {realized_shift_fraction:.3f}  '
        f'(target {shift_probability:.2f})'
    )
    axes[1].legend(fontsize=8, frameon=False)

    plt.tight_layout()
    plt.savefig('hourglass_epoch_sampling.png', dpi=150, bbox_inches='tight')
    plt.show()


def validate_epoch_sampling(phot_table, shift_probability=0.20, sample_size=2000,
                            random_seed=0, plot=True):
    random_generator = np.random.default_rng(random_seed)

    detection_epochs_per_object = (
        phot_table[phot_table['token_type'] == 'd'].groupby('cid')['mjd'].nunique()
    )
    eligible_cids = detection_epochs_per_object[detection_epochs_per_object >= 2].index.to_numpy()
    sample_cids = random_generator.choice(
        eligible_cids, size=min(sample_size, len(eligible_cids)), replace=False
    )

    grouped = dict(tuple(phot_table[phot_table['cid'].isin(sample_cids)].groupby('cid')))
    shifted_flags = []
    window_token_counts = []
    for cid in sample_cids:
        window_rows, anchor_mjd, was_shifted = sample_epoch_window(
            grouped[cid], shift_probability=shift_probability, random_generator=random_generator,
        )
        shifted_flags.append(was_shifted)
        window_token_counts.append(len(window_rows))
    realized_shift_fraction = float(np.mean(shifted_flags))

    if plot:
        plot_epoch_sampling(detection_epochs_per_object, realized_shift_fraction,
                            shift_probability, window_token_counts)
    return detection_epochs_per_object, realized_shift_fraction


detection_epochs_per_object, realized_shift_fraction = validate_epoch_sampling(
    phot, plot=True, random_seed=BATCH_SEED)


In [ ]:
from matplotlib.patches import FancyBboxPatch

EPOCH_GRID_BANDS = SURVEY_BANDS['WIDE']  # R, Z, Y, J — rows of the token grid, top-to-bottom


def select_token_diagram_object(phot_table, objects_path=OBJECTS_PATH):
    """First WIDE-survey SN Ia in `phot_table` whose 3-epoch window (unshifted) has >= 2
    detection visits and at least one upper-limit ('u') and one not-observed ('n') token.
    Deterministic: candidate cids are scanned in their order of appearance in phot_table.
    Returns (cid, window_rows, anchor_mjd, redshift).
    """
    objects_df = pq.read_table(objects_path, columns=['cid', 'class', 'field', 'z_cmb']).to_pandas()
    objects_df['survey'] = objects_df['field'].str.split('+').str[0]
    wide_sn_ia = objects_df[(objects_df['class'] == 'SN_Ia') & (objects_df['survey'] == 'WIDE')]
    redshift_by_cid = wide_sn_ia.set_index('cid')['z_cmb']
    wide_sn_ia_cids = set(wide_sn_ia['cid'])

    for cid in phot_table['cid'].drop_duplicates():
        if cid not in wide_sn_ia_cids:
            continue
        object_table = phot_table[phot_table['cid'] == cid]
        window_rows, anchor_mjd, _ = sample_epoch_window(object_table, shift_probability=0.0)
        n_detection_visits = window_rows.loc[window_rows['token_type'] == 'd', 'mjd'].nunique()
        has_upper_limit = (window_rows['token_type'] == 'u').any()
        has_not_observed = (window_rows['token_type'] == 'n').any()
        if n_detection_visits >= 2 and has_upper_limit and has_not_observed:
            return int(cid), window_rows, anchor_mjd, float(redshift_by_cid[cid])

    raise ValueError('No WIDE SN Ia satisfies the d>=2 / u / n condition.')


def _draw_light_curve_panel(axis, window_rows, epoch_mjds):
    for column_index, epoch_mjd in enumerate(epoch_mjds):
        axis.axvspan(epoch_mjd - 1.0, epoch_mjd + 1.0, color='0.5', alpha=0.08, zorder=0)
        axis.text(
            epoch_mjd, 1.01, f'epoch {column_index + 1}',
            transform=axis.get_xaxis_transform(), ha='center', va='bottom', fontsize=11, color='0.4',
        )

    measured = window_rows[window_rows['token_type'].isin(['d', 'u'])]
    measured = measured[measured['mag'].notna()]
    faintest_mag = measured['mag'].max() if len(measured) > 0 else 26.0
    # not-observed foot strip sits 0.4 mag below the faintest measured point (the upper
    # limit) — the detection limit itself is read off the upper-limit triangles, no extra line
    strip_mag = faintest_mag + 0.4

    for band in EPOCH_GRID_BANDS:
        color = BAND_COLORS[band]
        band_rows = window_rows[window_rows['band'] == band]

        detections = band_rows[band_rows['token_type'] == 'd']
        if len(detections) > 0:
            axis.errorbar(
                detections['mjd'], detections['mag'], yerr=detections['sigma_mag'],
                fmt=TOKEN_MARKERS['d'], color=color, markersize=5, markeredgecolor='black',
                markeredgewidth=0.4, lw=1.0, capsize=2, alpha=0.85,
            )

        upper_limits = band_rows[band_rows['token_type'] == 'u']
        if len(upper_limits) > 0:
            axis.scatter(
                upper_limits['mjd'], upper_limits['mag'], marker=TOKEN_MARKERS['u'],
                color=color, s=45, edgecolor='black', linewidth=0.4, alpha=0.5,
            )

    # not-observed bands: one square per missing band on that visit, stacked downward in a
    # foot strip (same convention as plot_token_light_curves)
    square_spacing = 0.25
    not_observed = window_rows[window_rows['token_type'] == 'n']
    for visit_mjd, visit_group in not_observed.groupby('mjd'):
        missing_bands = [band for band in EPOCH_GRID_BANDS if band in visit_group['band'].values]
        for stack_index, band in enumerate(missing_bands):
            axis.scatter(
                visit_mjd, strip_mag + stack_index * square_spacing,
                marker=TOKEN_MARKERS['n'], facecolor=BAND_COLORS[band],
                edgecolor='black', linewidth=0.4, s=28, alpha=0.7,
            )

    axis.invert_yaxis()
    axis.set_xlabel('MJD')
    axis.set_ylabel('magnitude')
    axis.set_title('Observed light curve', pad=24)
    band_handles = [
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=BAND_COLORS[band],
                   markersize=8, label=band)
        for band in EPOCH_GRID_BANDS
    ]
    axis.legend(handles=band_handles, frameon=False, fontsize=9, loc='best')


def _draw_token_grid_panel(axis, window_rows, epoch_mjds, delta_times, redshift):
    token_lookup = {
        (row['mjd'], row['band']): row
        for _, row in window_rows.iterrows()
    }
    n_rows = len(EPOCH_GRID_BANDS)
    n_columns = len(epoch_mjds)

    # cells are wider than tall so the "mag±σ" string fits inside the box
    cell_width, cell_height = 1.7, 1.0
    pad = 0.08

    for column_index, epoch_mjd in enumerate(epoch_mjds):
        for row_index, band in enumerate(EPOCH_GRID_BANDS):
            x = column_index * cell_width
            y = (n_rows - 1 - row_index) * cell_height  # R on top
            center_x, center_y = x + cell_width / 2, y + cell_height / 2
            token = token_lookup.get((epoch_mjd, band))
            token_type = token['token_type'] if token is not None else 'n'

            # the border always carries the filter colour (detection, upper limit and gap
            # alike); white fill, black text. The content tells the type apart:
            #   d -> mag±σ,  u -> <mag,  n -> ×
            axis.add_patch(FancyBboxPatch(
                (x + pad, y + pad), cell_width - 2 * pad, cell_height - 2 * pad,
                boxstyle='round,pad=0,rounding_size=0.06',
                facecolor='white', edgecolor=BAND_COLORS[band], linewidth=2.0, zorder=2,
            ))

            if token_type == 'd':
                axis.text(center_x, center_y, f"{token['mag']:.1f}±{token['sigma_mag']:.1f}",
                          ha='center', va='center', color='black', fontsize=10, zorder=3)
            elif token_type == 'u':
                axis.text(center_x, center_y, f"<{token['mag']:.1f}",
                          ha='center', va='center', color='black', fontsize=10.5, zorder=3)
            else:
                axis.text(center_x, center_y, '×', ha='center', va='center',
                          color='black', fontsize=18, zorder=3)

    grid_top = n_rows * cell_height
    for column_index, delta_time in enumerate(delta_times):
        axis.text(column_index * cell_width + cell_width / 2, grid_top + 0.12,
                  f'epoch {column_index + 1}\nΔt={delta_time:.0f} d',
                  ha='center', va='bottom', fontsize=10.5)
    for row_index, band in enumerate(EPOCH_GRID_BANDS):
        axis.text(-0.18, (n_rows - 1 - row_index) * cell_height + cell_height / 2, band,
                  ha='right', va='center', fontsize=12, fontweight='bold')

    # global [Z] token: one per object, sits beside the per-visit set (not inside any
    # magnitude token) — separated from the grid by a dashed rule
    grid_right = n_columns * cell_width
    axis.plot([grid_right + 0.22, grid_right + 0.22], [0.1, grid_top - 0.1],
              color='0.8', lw=0.8, ls='--', zorder=1)
    z_x = grid_right + 0.55
    z_y = grid_top / 2 - cell_height / 2
    axis.add_patch(FancyBboxPatch(
        (z_x + pad, z_y + pad), cell_width - 2 * pad, cell_height - 2 * pad,
        boxstyle='round,pad=0,rounding_size=0.06',
        facecolor='white', edgecolor='0.3', linewidth=2.0, zorder=2,
    ))
    axis.text(z_x + cell_width / 2, z_y + cell_height / 2, f'[Z]\nz={redshift:.2f}',
              ha='center', va='center', color='black', fontsize=10.5, zorder=3)
    axis.text(z_x + cell_width / 2, grid_top + 0.12, 'global\ntoken',
              ha='center', va='bottom', fontsize=10.5)

    axis.set_xlim(-1.0, z_x + cell_width + 0.3)
    axis.set_ylim(-0.3, grid_top + 1.0)
    axis.set_aspect('equal')
    axis.axis('off')
    axis.set_title('Tokenised input to transformer', pad=22)


def _draw_exploded_token(axis, token, delta_time):
    # a single per-visit token: (Δt, band) + the magnitude channel (mag, σ_mag) + the
    # detected/upper-limit/not-observed mask. Redshift is NOT carried here — it enters once
    # as the global [Z] token (drawn in the grid panel), never per magnitude token.
    fields = [
        ('Δt', f'{delta_time:.0f} d'),
        ('band', str(token['band'])),
        ('mask', 'det'),
        ('mag', f"{token['mag']:.1f}"),
        ('σ_mag', f"{token['sigma_mag']:.2f}"),
    ]
    box_width, box_height, gap = 1.0, 0.6, 0.18
    y_bottom = 0.25
    center_y = y_bottom + box_height / 2

    for field_index, (field_name, field_value) in enumerate(fields):
        x = field_index * (box_width + gap)
        axis.add_patch(FancyBboxPatch(
            (x, y_bottom), box_width, box_height,
            boxstyle='round,pad=0,rounding_size=0.05',
            facecolor='#f0f0f0', edgecolor='0.4', linewidth=0.8,
        ))
        axis.text(x + box_width / 2, y_bottom + box_height + 0.08, field_name,
                  ha='center', va='bottom', fontsize=10, color='0.3')
        axis.text(x + box_width / 2, center_y, field_value,
                  ha='center', va='center', fontsize=11)
        if field_index < len(fields) - 1:
            axis.annotate('', xy=(x + box_width + gap, center_y), xytext=(x + box_width, center_y),
                          arrowprops=dict(arrowstyle='->', color='0.5', lw=0.6))

    fields_right = len(fields) * (box_width + gap) - gap
    token_x = fields_right + 0.5
    axis.annotate('', xy=(token_x, center_y), xytext=(fields_right, center_y),
                  arrowprops=dict(arrowstyle='-|>', color='0.2', lw=1.2))
    token_box_width = 1.6
    axis.add_patch(FancyBboxPatch(
        (token_x, y_bottom - 0.05), token_box_width, box_height + 0.1,
        boxstyle='round,pad=0,rounding_size=0.05',
        facecolor='#dce7f5', edgecolor='0.25', linewidth=1.0,
    ))
    axis.text(token_x + token_box_width / 2, center_y, 'token',
              ha='center', va='center', fontsize=12)
    axis.text((token_x + token_box_width) / 2, y_bottom + box_height + 0.24,
              'per-visit token  (one per band × epoch)',
              ha='center', va='bottom', fontsize=10, color='0.35', style='italic')

    axis.set_xlim(-0.2, token_x + token_box_width + 0.2)
    axis.set_ylim(0, 1.15)
    axis.axis('off')


def plot_token_diagram(phot_table, plot=True):
    """Two-panel figure: a real WIDE SN Ia light curve (3-epoch window) and the token grid it
    becomes as transformer input, with one exploded token showing the per-field encoding.
    """
    cid, window_rows, anchor_mjd, redshift = select_token_diagram_object(phot_table)
    epoch_mjds = np.sort(window_rows.loc[window_rows['token_type'] == 'd', 'mjd'].unique())
    delta_times = epoch_mjds - epoch_mjds[0]

    if not plot:
        return cid, window_rows

    figure = plt.figure(figsize=(13.5, 6.5))
    grid_spec = figure.add_gridspec(2, 2, width_ratios=[55, 45], height_ratios=[3, 1.4],
                                    hspace=0.35, wspace=0.15)
    axis_light_curve = figure.add_subplot(grid_spec[:, 0])
    axis_token_grid = figure.add_subplot(grid_spec[0, 1])
    axis_exploded = figure.add_subplot(grid_spec[1, 1])

    _draw_light_curve_panel(axis_light_curve, window_rows, epoch_mjds)
    _draw_token_grid_panel(axis_token_grid, window_rows, epoch_mjds, delta_times, redshift)

    example_token = window_rows[window_rows['token_type'] == 'd'].sort_values('mjd').iloc[0]
    example_delta_time = example_token['mjd'] - epoch_mjds[0]
    _draw_exploded_token(axis_exploded, example_token, example_delta_time)

    plt.savefig('hourglass_token_diagram.png', dpi=200, bbox_inches='tight')
    plt.show()
    return cid, window_rows


diagram_cid, diagram_window_rows = plot_token_diagram(phot, plot=True)


## From pandas to PyTorch — building the token tensor

The cells above kept everything as a pandas table: one row per band×visit, with `mag`, `sigma_mag` and a `token_type` ∈ {`d`, `u`, `n`}. The transformer doesn't read a DataFrame — it reads **tensors**. This cell does the conversion for one object's sampled window.

What "defining the token" means in practice: each pandas row becomes one **token**, a fixed set of numeric channels. We split each row's information into channels the model can embed separately:

- **`delta_time`** — days since first detection (`mjd − anchor_mjd`). Stays continuous; the time2vec encoding happens *inside* the model, not here.
- **`band_index`, `token_type_index`** — the categorical fields (`R/Z/Y/J/H/F` and `d/u/n`) become integer indices. The model turns each integer into a learned embedding vector.
- **`magnitude`, `sigma_magnitude`** — the continuous photometry, **globally normalized** (subtract a single train-set mean, divide by a single std). Not per object: how bright the source *is* is signal we must keep.
- **`magnitude_mask`, `sigma_mask`** — because `n` tokens have no magnitude and `u` tokens have no error, those channels are set to 0 after normalization; the mask flag is what tells the model "this 0 is *absent*, not a real value".

The redshift is **not** a per-visit channel — it's one **global `[Z]` token** per object: `(redshift, redshift_error)` plus a `has_redshift` flag (the switch to the learned `[NO_Z]` token when there is no z).

The figure prints the actual tensor: rows are tokens, columns are the channels. Read across a row to see what one band×visit became; the integer columns (`band_index`, `type_index`, the masks) and the normalized float columns are exactly what the embedding layers will consume. Batching several objects together (padding to equal length + a padding mask) is left to the dataloader — here we define a single object's token tensor.


In [ ]:
import torch

BAND_TO_INDEX = {band: index for index, band in enumerate(BAND_ORDER)}
TOKEN_TYPE_TO_INDEX = {'d': 0, 'u': 1, 'n': 2}

# Global magnitude normalization: mean/std fit ONCE on every detection in the training
# table, then reused for upper-limit magnitudes and (later) for injected KNe. Never per
# object — apparent brightness is signal, normalizing it away per object would erase it.
_detection_rows = phot[phot['token_type'] == 'd']
MAG_MEAN = float(_detection_rows['mag'].mean())
MAG_STD = float(_detection_rows['mag'].std())
SIGMA_MAG_MEAN = float(_detection_rows['sigma_mag'].mean())
SIGMA_MAG_STD = float(_detection_rows['sigma_mag'].std())


def plot_token_tensor(tokens, window_rows, redshift):
    """Render the assembled per-visit token tensor as an annotated matrix: one row per
    token (band / type / Δt), one column per feature channel. Colour is column-wise min-max
    only to make the structure readable; the printed numbers are the actual tensor values
    the transformer will receive. The global [Z] token is annotated below the grid.
    """
    feature_names = ['Δt', 'band_index', 'type_index', 'mag_norm', 'σ_mag_norm', 'mag_mask', 'σ_mask']
    matrix = np.stack(
        [
            tokens['delta_time'].numpy(),
            tokens['band_index'].numpy().astype(np.float32),
            tokens['token_type_index'].numpy().astype(np.float32),
            tokens['magnitude'].numpy(),
            tokens['sigma_magnitude'].numpy(),
            tokens['magnitude_mask'].numpy(),
            tokens['sigma_mask'].numpy(),
        ],
        axis=1,
    )
    n_tokens, n_features = matrix.shape

    column_normalized = np.zeros_like(matrix)
    for column_index in range(n_features):
        column = matrix[:, column_index]
        spread = column.max() - column.min()
        column_normalized[:, column_index] = (column - column.min()) / spread if spread > 0 else 0.5

    token_labels = [
        f'{band}  {token_type}  Δt={delta_time:.0f}'
        for band, token_type, delta_time in zip(
            window_rows['band'], window_rows['token_type'], tokens['delta_time'].numpy()
        )
    ]
    integer_columns = {1, 2, 5, 6}

    figure, axis = plt.subplots(figsize=(1.3 * n_features + 2, 0.5 * n_tokens + 2.5))
    axis.imshow(column_normalized, cmap='coolwarm', aspect='auto', alpha=0.75)

    for row_index in range(n_tokens):
        for column_index in range(n_features):
            value = matrix[row_index, column_index]
            text = f'{int(value)}' if column_index in integer_columns else f'{value:.2f}'
            axis.text(column_index, row_index, text, ha='center', va='center', fontsize=9)

    axis.set_xticks(range(n_features))
    axis.set_xticklabels(feature_names, rotation=30, ha='right')
    axis.set_yticks(range(n_tokens))
    axis.set_yticklabels(token_labels, fontsize=9)
    axis.set_title(
        f'Per-visit token tensor  ({n_tokens} tokens × {n_features} features)\n'
        f'global [Z] token:  z={redshift:.3f}   has_redshift={float(tokens["has_redshift"]):.0f}',
        fontsize=11,
    )
    plt.tight_layout()
    plt.savefig('hourglass_token_tensor.png', dpi=150, bbox_inches='tight')
    plt.show()


def encode_window_to_tokens(window_rows, anchor_mjd, redshift, redshift_error=np.nan, plot=True):
    """Convert one object's sampled visit window (a pandas slice) into the PyTorch tensors
    the transformer consumes. One row of `window_rows` -> one per-visit token; the object's
    redshift -> one global [Z] token.

    Per-visit channels (each a length-`n_tokens` tensor):
      delta_time        Δt since first detection, in days — continuous; time2vec lives in the model
      band_index        categorical index into BAND_ORDER          -> band embedding (model)
      token_type_index  0=d detection, 1=u upper limit, 2=n gap    -> type embedding (model)
      magnitude         globally-normalized mag; 0 where masked (n tokens carry no magnitude)
      sigma_magnitude   globally-normalized σ_mag; 0 where masked (u and n carry no error)
      magnitude_mask    1 if the magnitude channel carries information (d, u), else 0
      sigma_mask        1 if σ_mag carries information (d only), else 0

    The masks let the model distinguish a genuine 0 from an absent channel after
    normalization. The [Z] token is global (one per object, not per visit): (redshift,
    redshift_error) + a has_redshift flag the model maps to [Z] vs the learned [NO_Z] token.
    Returns a dict of tensors (no batching here — padding into a batch is the dataloader's job).
    """
    window_rows = window_rows.assign(
        _band_order=window_rows['band'].map(BAND_TO_INDEX)
    ).sort_values(['mjd', '_band_order']).drop(columns='_band_order')

    delta_time = (window_rows['mjd'] - anchor_mjd).to_numpy(dtype=np.float32)
    band_index = window_rows['band'].map(BAND_TO_INDEX).to_numpy(dtype=np.int64)
    token_type_index = window_rows['token_type'].map(TOKEN_TYPE_TO_INDEX).to_numpy(dtype=np.int64)

    raw_magnitude = window_rows['mag'].to_numpy(dtype=np.float32)
    raw_sigma_magnitude = window_rows['sigma_mag'].to_numpy(dtype=np.float32)
    magnitude_mask = ~np.isnan(raw_magnitude)        # d and u carry a magnitude, n does not
    sigma_mask = ~np.isnan(raw_sigma_magnitude)      # only d carries an error

    # observational-scenario channels (raw, un-normalized): they ride along like
    # true_redshift so the matched KN twin can be built at this visit/band/zp. NOT model
    # inputs (not in PER_TOKEN_KEYS) and never normalized. NaN on 'n' tokens.
    flux_true = window_rows['flux_true'].to_numpy(dtype=np.float32)
    fluxcal_err = window_rows['fluxcal_err'].to_numpy(dtype=np.float32)
    zero_point = window_rows['zp'].to_numpy(dtype=np.float32)

    magnitude = np.where(magnitude_mask, (raw_magnitude - MAG_MEAN) / MAG_STD, 0.0).astype(np.float32)
    sigma_magnitude = np.where(
        sigma_mask, (raw_sigma_magnitude - SIGMA_MAG_MEAN) / SIGMA_MAG_STD, 0.0
    ).astype(np.float32)

    has_redshift = bool(np.isfinite(redshift))

    tokens = {
        'delta_time': torch.from_numpy(delta_time),
        'band_index': torch.from_numpy(band_index),
        'token_type_index': torch.from_numpy(token_type_index),
        'magnitude': torch.from_numpy(magnitude),
        'sigma_magnitude': torch.from_numpy(sigma_magnitude),
        'magnitude_mask': torch.from_numpy(magnitude_mask.astype(np.float32)),
        'sigma_mask': torch.from_numpy(sigma_mask.astype(np.float32)),
        'redshift': torch.tensor(redshift if has_redshift else 0.0, dtype=torch.float32),
        'redshift_error': torch.tensor(
            redshift_error if np.isfinite(redshift_error) else 0.0, dtype=torch.float32
        ),
        'has_redshift': torch.tensor(float(has_redshift), dtype=torch.float32),
        'flux_true': torch.from_numpy(flux_true),
        'fluxcal_err': torch.from_numpy(fluxcal_err),
        'zp': torch.from_numpy(zero_point),
    }

    if plot:
        plot_token_tensor(tokens, window_rows, redshift)
    return tokens


example_object = token_window_selection[0]
example_tokens = encode_window_to_tokens(
    example_object['window_rows'],
    anchor_mjd=example_object['anchor_mjd'],
    redshift=example_object['z_cmb'],
    plot=True,
)
{name: tuple(tensor.shape) for name, tensor in example_tokens.items()}


## Train / val / test split — grouped by `cid`

### Target classes

The plan's label scheme is **`{Ia, II, other, KN}`**. We build it from the Hourglass classes like this:

- **`Ia`** ← `SN_Ia`, **`II`** ← `CCSN` — the two dominant, well-populated transient families.
- **`other`** ← everything else that *is* a transient (`SN_Iax`, `SNIa-91bg`, `SLSN-I`, `TDE`, `ILOT`, `PISN`), lumped into one class: individually too rare to learn on their own, but real contaminants the model must not mistake for a KN.
- **Dropped entirely**: `AGN` and `Fixed_mag` (not transients — they don't belong in a transient pipeline), and the handful of native `KN`. The `KN` label is **reserved (index 3)** for the *injected* kilonovae added in the generation step; the few KN that happen to live in Hourglass are not the signal we train on and would only contaminate it.

So today the contaminant catalog yields **three** populated classes (`Ia`, `II`, `other`); the fourth (`KN`) stays empty until injection.

### How we split

Two rules drive the split:

1. **Split by object, never by row.** Each physical object (`cid`) owns several rows (band × visit) and, later, several *augmented* windows of itself. If the same `cid` had rows in both train and test, the model could memorize that object during training and "recognize" it at test time — the test score would be optimistic and meaningless. So the unit we split is the **`cid`**, and every row / window of a given `cid` travels with it into exactly one of the three sets. This is a **grouped split**.

2. **Stratify by target class.** Class proportions are wildly uneven (SN Ia and CCSN dominate, the `other` members are rare). A blind random split could leave a rare class almost absent from validation. We therefore split **within each target class**: shuffle that class's `cid`s, then carve the 70 / 15 / 15 fractions. Each split ends up with the same class mix as the whole catalog, and the grouping guarantee from rule 1 still holds because we only ever move whole `cid`s.

The result is three **disjoint sets of `cid`s**. We verify disjointness explicitly — a silent leak here invalidates every number downstream.


In [ ]:
from torch.utils.data import Dataset, DataLoader

# {Ia, II, other, KN} per the plan. KN is reserved (index 3) for the injected signal added
# later; the native Hourglass KN are dropped together with the non-transients AGN/Fixed_mag.
DROP_CLASSES = ('AGN', 'Fixed_mag', 'KN')
CLASS_TO_GROUP = {
    'SN_Ia': 'Ia',
    'CCSN': 'II',
    'SN_Iax': 'other',
    'SNIa-91bg': 'other',
    'SLSN-I': 'other',
    'TDE': 'other',
    'ILOT': 'other',
    'PISN': 'other',
}
GROUP_ORDER = ['Ia', 'II', 'other', 'KN']
GROUP_TO_LABEL = {group_name: index for index, group_name in enumerate(GROUP_ORDER)}

surviving_cids = phot['cid'].unique()
objects_meta = pq.read_table(OBJECTS_PATH, columns=['cid', 'class', 'z_cmb']).to_pandas()
objects_meta = objects_meta[objects_meta['cid'].isin(surviving_cids)]
objects_meta = objects_meta[~objects_meta['class'].isin(DROP_CLASSES)].set_index('cid')
objects_meta['group'] = objects_meta['class'].map(CLASS_TO_GROUP)

kept_cids = objects_meta.index.to_numpy()
group_per_cid = objects_meta['group']
redshift_by_cid = objects_meta['z_cmb'].to_dict()
label_by_cid = objects_meta['group'].map(GROUP_TO_LABEL).to_dict()


def split_cids_by_class(group_per_cid, fractions=(0.70, 0.15, 0.15), random_seed=42):
    """Stratified grouped split: shuffle each target class's cids independently and carve the
    train / val / test fractions inside the class, so the unit moved is always a whole cid
    (no row-level leakage) and every split keeps the catalog's class proportions.
    Returns three numpy arrays of cids.
    """
    random_generator = np.random.default_rng(random_seed)
    train_fraction, validation_fraction, _ = fractions

    train_cids = []
    validation_cids = []
    test_cids = []
    for group_name, class_group in group_per_cid.groupby(group_per_cid):
        cids = class_group.index.to_numpy().copy()
        random_generator.shuffle(cids)
        n_objects = len(cids)
        n_train = int(round(train_fraction * n_objects))
        n_validation = int(round(validation_fraction * n_objects))

        train_cids.extend(cids[:n_train])
        validation_cids.extend(cids[n_train:n_train + n_validation])
        test_cids.extend(cids[n_train + n_validation:])

    return np.array(train_cids), np.array(validation_cids), np.array(test_cids)


train_cids, validation_cids, test_cids = split_cids_by_class(group_per_cid)

# disjointness is the whole point — assert it rather than trust it
assert len(set(train_cids) & set(validation_cids)) == 0
assert len(set(train_cids) & set(test_cids)) == 0
assert len(set(validation_cids) & set(test_cids)) == 0
assert len(train_cids) + len(validation_cids) + len(test_cids) == len(kept_cids)

split_summary = pd.DataFrame({
    'train': group_per_cid.loc[train_cids].value_counts(),
    'val': group_per_cid.loc[validation_cids].value_counts(),
    'test': group_per_cid.loc[test_cids].value_counts(),
}).reindex(GROUP_ORDER).fillna(0).astype(int)
split_summary.loc['total'] = split_summary.sum()
print(f'train {len(train_cids):,}   val {len(validation_cids):,}   test {len(test_cids):,}')
split_summary


## Refit the magnitude normalization on the training split only

The token tensor cell above fit `MAG_MEAN / MAG_STD / SIGMA_MAG_MEAN / SIGMA_MAG_STD` on **every** detection in the catalog — convenient while prototyping a single object, but a leak now that the split exists: the normalization constants would carry information from validation and test magnitudes into training.

The rule is the standard one — **all preprocessing statistics are fit on the training set and merely *applied* to val/test** (and, later, reused unchanged for the injected KNe). We recompute the four constants over the training `cid`s' detections and overwrite the module-level globals, so `encode_window_to_tokens` (which reads them) now normalizes with train-only stats everywhere it is called.


In [ ]:
# refit global magnitude normalization on TRAIN detections only, then reuse for val/test/KNe
_train_detection_rows = phot[(phot['token_type'] == 'd') & (phot['cid'].isin(train_cids))]
MAG_MEAN = float(_train_detection_rows['mag'].mean())
MAG_STD = float(_train_detection_rows['mag'].std())
SIGMA_MAG_MEAN = float(_train_detection_rows['sigma_mag'].mean())
SIGMA_MAG_STD = float(_train_detection_rows['sigma_mag'].std())

print(f'mag:        mean {MAG_MEAN:.3f}  std {MAG_STD:.3f}')
print(f'sigma_mag:  mean {SIGMA_MAG_MEAN:.3f}  std {SIGMA_MAG_STD:.3f}')


## The `Dataset` and the `DataLoader`

PyTorch separates two jobs:

- A **`Dataset`** knows how to produce **one** example given an index: here, "take the `index`-th `cid`, sample its token window, encode it to the per-visit token tensors". This reuses the two functions we already built — `sample_epoch_window` and `encode_window_to_tokens` — unchanged. We pre-group `phot` by `cid` into a dictionary once, so `__getitem__` is a dictionary lookup plus the encoding, never a scan of the 1.7 M-row table.
  - A single **`data_aug`** flag turns on the three training-time augmentations, drawn fresh on every `__getitem__` (stochastic per epoch, not a fixed split): **window shift** (`p = SHIFT_PROBABILITY = 0.20`, slide to epochs `{2,3,4}`), **prefix truncation** (reveal only the first 1 / 2 / 3 detection epochs, equal probability — the early-classification regime), and **redshift dropout** (`p = REDSHIFT_DROPOUT_PROBABILITY = 0.5`, hide z from the model → the learned `[NO_Z]` token). Train uses `data_aug=True`; val/test use `data_aug=False` (full window, true z). The **true redshift is always kept in `true_redshift`**, even when z is dropped from the model input, so the second (KN) dataloader can build the matched twin at this same z.

- A **`DataLoader`** batches several `Dataset` outputs together and hands them to the training loop. This is where the one remaining problem is solved: **objects have different token counts** (a different number of bands detected, upper limits, gaps, and now a variable number of visible epochs). A tensor batch must be rectangular, so the `collate_fn` **pads** every object up to the batch's longest token sequence and builds a **`padding_mask`** — `True` where a position is padding. The transformer's attention will read that mask and ignore the padded slots. Without it, the zeros we pad with would look like real `band 0`, `type d` tokens. The global `[Z]` fields (one scalar per object) need no padding — they just stack.

The last cell pulls one training batch and prints the shapes: per-token tensors are `(batch_size, max_tokens)`, the mask is `(batch_size, max_tokens)`, and the global / label / `true_redshift` tensors are `(batch_size,)`.


In [ ]:
PER_TOKEN_KEYS = [
    'delta_time', 'band_index', 'token_type_index',
    'magnitude', 'sigma_magnitude', 'magnitude_mask', 'sigma_mask',
]
GLOBAL_KEYS = ['redshift', 'redshift_error', 'has_redshift']
# Per-token observational-scenario channels for the matched KN twin. Padded like
# PER_TOKEN_KEYS but kept OUT of the model feature stack (not normalized, not embedded):
#   flux_true    noiseless contaminant model flux [fluxcal] (from sim_mag_obs)
#   fluxcal_err  contaminant flux error [fluxcal] (sigma_cont)
#   zp           per-visit instrumental zeropoint -> alpha = 10^((27.5 - zp)/2.5)
SCENARIO_KEYS = ['flux_true', 'fluxcal_err', 'zp']

# group the long table once: cid -> that object's rows. __getitem__ becomes a dict lookup.
photometry_by_cid = dict(tuple(phot.groupby('cid')))

SHIFT_PROBABILITY = 0.20             # slide the window to {2,3,4} (late-onset simulation)
REDSHIFT_DROPOUT_PROBABILITY = 0.50  # hide z from the model -> learned [NO_Z] token


def truncate_to_visible_epochs(window_rows, random_generator, max_epochs=EPOCHS_PER_WINDOW):
    """Random prefix truncation for early classification: reveal only the first k visit-day
    epochs, k ~ Uniform{1, ..., max_epochs}, capped at the epochs the object actually has.
    Every token after the k-th visit day is dropped (the model sees only what has been
    observed so far); within the visible epochs all d/u/n tokens are kept. Equal probability
    of 1 / 2 / 3 visible epochs.
    """
    epoch_mjds = np.sort(window_rows['mjd'].unique())
    target_epochs = int(random_generator.integers(1, max_epochs + 1))
    visible_epochs = min(target_epochs, len(epoch_mjds))
    visible_mjds = epoch_mjds[:visible_epochs]
    return window_rows[window_rows['mjd'].isin(visible_mjds)]


class HourglassWindowDataset(Dataset):
    """One example = one object's sampled token window (the tensors built by
    encode_window_to_tokens) plus its integer class label and its true redshift.

    `data_aug=True` (training) turns on three independent, dataloader-time augmentations,
    fresh on every draw: window shift (p=SHIFT_PROBABILITY), prefix truncation to 1/2/3
    epochs, and redshift dropout (p=REDSHIFT_DROPOUT_PROBABILITY). `data_aug=False`
    (val/test) feeds the full 3-epoch window with the true z.

    This is the *survey* dataloader (Roman / Hourglass contaminants). The true redshift is
    always kept in `true_redshift` — even when z is dropped from the model input — so the
    second (KN) dataloader can later build the matched twin at this z.
    """

    def __init__(self, cids, photometry_by_cid, redshift_by_cid, label_by_cid,
                 data_aug=False, random_seed=None):
        self.cids = list(cids)
        self.photometry_by_cid = photometry_by_cid
        self.redshift_by_cid = redshift_by_cid
        self.label_by_cid = label_by_cid
        self.data_aug = data_aug
        self.random_generator = np.random.default_rng(random_seed)

    def __len__(self):
        return len(self.cids)

    def __getitem__(self, index):
        cid = self.cids[index]
        shift_probability = SHIFT_PROBABILITY if self.data_aug else 0.0
        window_rows, anchor_mjd, _ = sample_epoch_window(
            self.photometry_by_cid[cid],
            shift_probability=shift_probability,
            random_generator=self.random_generator,
        )
        if self.data_aug:
            window_rows = truncate_to_visible_epochs(window_rows, self.random_generator)

        true_redshift = self.redshift_by_cid[cid]
        # z-dropout is symmetric across classes (see plan.md): drop only the z the MODEL sees,
        # never the true z — that one travels on for the KN twin built at this same redshift.
        redshift_for_input = true_redshift
        if self.data_aug and self.random_generator.random() < REDSHIFT_DROPOUT_PROBABILITY:
            redshift_for_input = np.nan

        tokens = encode_window_to_tokens(
            window_rows, anchor_mjd=anchor_mjd, redshift=redshift_for_input, plot=False,
        )
        tokens['label'] = torch.tensor(self.label_by_cid[cid], dtype=torch.long)
        tokens['cid'] = torch.tensor(int(cid), dtype=torch.long)
        tokens['true_redshift'] = torch.tensor(
            true_redshift if np.isfinite(true_redshift) else np.nan, dtype=torch.float32,
        )
        return tokens


def collate_token_windows(batch):
    """Pad a list of variable-length token dicts into rectangular batch tensors and build
    the padding mask (True = padded slot the transformer must ignore). Globals just stack.
    `true_redshift` rides along (not a model input) so the KN dataloader can read it.
    """
    batch_size = len(batch)
    max_tokens = max(item['delta_time'].shape[0] for item in batch)

    padded = {
        key: torch.zeros(batch_size, max_tokens, dtype=batch[0][key].dtype)
        for key in PER_TOKEN_KEYS + SCENARIO_KEYS
    }
    padding_mask = torch.ones(batch_size, max_tokens, dtype=torch.bool)

    for row_index, item in enumerate(batch):
        n_tokens = item['delta_time'].shape[0]
        for key in PER_TOKEN_KEYS + SCENARIO_KEYS:
            padded[key][row_index, :n_tokens] = item[key]
        padding_mask[row_index, :n_tokens] = False

    collated = dict(padded)
    collated['padding_mask'] = padding_mask
    for key in GLOBAL_KEYS:
        collated[key] = torch.stack([item[key] for item in batch])
    collated['label'] = torch.stack([item['label'] for item in batch])
    collated['cid'] = torch.stack([item['cid'] for item in batch])
    collated['true_redshift'] = torch.stack([item['true_redshift'] for item in batch])
    return collated


BATCH_SIZE = 64

train_dataset = HourglassWindowDataset(
    train_cids, photometry_by_cid, redshift_by_cid, label_by_cid, data_aug=True, random_seed=0)
validation_dataset = HourglassWindowDataset(
    validation_cids, photometry_by_cid, redshift_by_cid, label_by_cid, data_aug=False)
test_dataset = HourglassWindowDataset(
    test_cids, photometry_by_cid, redshift_by_cid, label_by_cid, data_aug=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_token_windows)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_token_windows)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_token_windows)

example_batch = next(iter(train_loader))
print('one training batch (data_aug=True):')
for key, tensor in example_batch.items():
    print(f'  {key:18s} {tuple(tensor.shape)}  {tensor.dtype}')


In [ ]:
# ## Sanity check — reconstruct the 3-epoch light curve straight from the batch tensor
#
# The dataloader hands the model normalized integers and floats; this cell proves the
# round-trip is intact by *inverting* it — denormalize the magnitude channels, map the
# integer `band_index` / `token_type_index` back to `R/Z/Y/J/H/F` and `d/u/n`, drop the
# padded slots via `padding_mask`, and redraw one object exactly in the style of
# `plot_token_light_curves`. If the reconstructed window matches the raw light curve, the
# encode -> collate -> pad path carried no silent corruption.

INDEX_TO_BAND = {index: band for band, index in BAND_TO_INDEX.items()}
INDEX_TO_TOKEN_TYPE = {index: token_type for token_type, index in TOKEN_TYPE_TO_INDEX.items()}


def reconstruct_window_from_tensor(batch, index_in_batch=0, plot=True):
    """Invert the dataloader encoding for one object in a collated batch and (optionally)
    redraw its 3-epoch window. Reads only what the model sees — the per-token tensors,
    the padding mask and the global [Z] fields — denormalizes mag / sigma_mag with the
    train-fit constants and reverses the categorical index maps. Returns the recovered
    per-token DataFrame (Δt, band, token_type, mag, sigma_mag) for inspection.
    """
    keep = ~batch['padding_mask'][index_in_batch].numpy()

    delta_time = batch['delta_time'][index_in_batch].numpy()[keep]
    band_index = batch['band_index'][index_in_batch].numpy()[keep]
    token_type_index = batch['token_type_index'][index_in_batch].numpy()[keep]
    magnitude_mask = batch['magnitude_mask'][index_in_batch].numpy()[keep].astype(bool)
    sigma_mask = batch['sigma_mask'][index_in_batch].numpy()[keep].astype(bool)

    magnitude = batch['magnitude'][index_in_batch].numpy()[keep] * MAG_STD + MAG_MEAN
    sigma_magnitude = batch['sigma_magnitude'][index_in_batch].numpy()[keep] * SIGMA_MAG_STD + SIGMA_MAG_MEAN

    # NaN back into the masked slots so 'n' tokens carry no magnitude and 'u' no error,
    # matching the convention add_token_magnitudes / encode_window_to_tokens started from
    magnitude = np.where(magnitude_mask, magnitude, np.nan)
    sigma_magnitude = np.where(sigma_mask, sigma_magnitude, np.nan)

    recovered = pd.DataFrame({
        'delta_time': delta_time,
        'band': [INDEX_TO_BAND[index] for index in band_index],
        'token_type': [INDEX_TO_TOKEN_TYPE[index] for index in token_type_index],
        'mag': magnitude,
        'sigma_mag': sigma_magnitude,
    })

    label_index = int(batch['label'][index_in_batch])
    cid = int(batch['cid'][index_in_batch])
    has_redshift = float(batch['has_redshift'][index_in_batch])
    redshift = float(batch['redshift'][index_in_batch])

    if plot:
        plot_reconstructed_window(recovered, cid, GROUP_ORDER[label_index], redshift, has_redshift)
    return recovered


def plot_reconstructed_window(recovered, cid, label_name, redshift, has_redshift):
    """Redraw a window reconstructed from the batch tensor — same markers/colors as
    plot_token_light_curves but on the Δt (days since first detection) axis the model uses.
    Circle: detection (mag ± σ), downward triangle: 5σ upper limit, square (foot strip):
    not-observed band.
    """
    measured = recovered[recovered['token_type'].isin(['d', 'u'])]
    measured = measured[measured['mag'].notna()]
    faintest_mag = measured['mag'].max() if len(measured) > 0 else 26.0
    strip_mag = faintest_mag + 0.8

    figure, axis = plt.subplots(figsize=(6.5, 4.0))

    for band in BAND_ORDER:
        color = BAND_COLORS[band]
        band_measured = measured[measured['band'] == band]

        detections = band_measured[band_measured['token_type'] == 'd']
        if len(detections) > 0:
            axis.errorbar(
                detections['delta_time'],
                detections['mag'],
                yerr=detections['sigma_mag'],
                fmt=TOKEN_MARKERS['d'],
                color=color,
                markersize=6,
                markeredgecolor='black',
                markeredgewidth=0.4,
                lw=1.0,
                capsize=2,
                alpha=0.85,
            )

        upper_limits = band_measured[band_measured['token_type'] == 'u']
        if len(upper_limits) > 0:
            axis.scatter(
                upper_limits['delta_time'],
                upper_limits['mag'],
                marker=TOKEN_MARKERS['u'],
                color=color,
                s=50,
                edgecolor='black',
                linewidth=0.4,
                alpha=0.5,
            )

    square_spacing = 0.25
    not_observed = recovered[recovered['token_type'] == 'n']
    for visit_delta_time, visit_group in not_observed.groupby('delta_time'):
        missing_bands = [band for band in BAND_ORDER if band in visit_group['band'].values]
        for stack_index, band in enumerate(missing_bands):
            axis.scatter(
                visit_delta_time,
                strip_mag + stack_index * square_spacing,
                marker=TOKEN_MARKERS['n'],
                facecolor=BAND_COLORS[band],
                edgecolor='black',
                linewidth=0.4,
                s=30,
                alpha=0.7,
            )

    axis.invert_yaxis()
    axis.set_xlabel(r'$\Delta t$ since first detection [days]', fontsize=10)
    axis.set_ylabel(r'mag$_{\rm calib}$ [mag]', fontsize=10)
    redshift_tag = f'z={redshift:.3f}' if has_redshift else 'no redshift'
    axis.set_title(
        f'reconstructed from batch tensor  |  cid={cid}  label={label_name}  {redshift_tag}',
        fontsize=10,
    )
    axis.grid(alpha=0.2)

    token_handles = [
        plt.Line2D([0], [0], marker=TOKEN_MARKERS[token_type], color='0.3', linestyle='none',
                   markersize=8, markeredgecolor='black', markeredgewidth=0.4, label=TOKEN_LABELS[token_type])
        for token_type in ['d', 'u', 'n']
    ]
    band_handles = [
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=BAND_COLORS[band],
                   markersize=7, label=band)
        for band in BAND_ORDER
    ]
    axis.legend(handles=token_handles + band_handles, loc='center left',
                bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=8)

    plt.tight_layout()
    plt.savefig('hourglass_window_from_tensor.png', dpi=150, bbox_inches='tight')
    plt.show()


reconstructed_window = reconstruct_window_from_tensor(example_batch, index_in_batch=0, plot=True)
reconstructed_window


In [ ]:
# ## Human-magnitude check — the light curve the model sees vs the scenario the KN twin gets
#
# Both data streams of one batched object, decoded back to apparent magnitude so an encoding
# error shows up by eye. Left block = what the transformer ingests (model channels). Right
# block = the raw scenario channels handed to the KN dataloader, plus two magnitude
# cross-checks against them.

def decode_batch_object_to_magnitudes(batch, index_in_batch=0):
    """One batched object back in human magnitudes for an end-to-end sanity check.

    Model channels (what the transformer ingests, denormalized):
      mag_model    apparent mag from the noisy fluxcal (NaN on 'n' tokens)
      sigma_model  its error (only 'd' carries one)
    Scenario channels (what the KN dataloader consumes) + magnitude cross-checks:
      mag_true     27.5 - 2.5 log10(flux_true): the NOISELESS contaminant mag (= sim_mag_obs).
                   For 'd' it must track mag_model within sigma_model.
      maglim_5sig  27.5 - 2.5 log10(5 * fluxcal_err): the per-visit 5 sigma depth. For 'u' it
                   must equal mag_model exactly (that is how the upper-limit token was encoded).
      flux_true, fluxcal_err, zp, alpha   the raw inputs to the matched-twin error:
                   sigma_KN^2 = fluxcal_err^2 + alpha * (F_KN - flux_true),  alpha = 10^((27.5 - zp)/2.5)
    """
    keep = ~batch['padding_mask'][index_in_batch].numpy()

    band = [INDEX_TO_BAND[band_index] for band_index in batch['band_index'][index_in_batch].numpy()[keep]]
    token_type = [
        INDEX_TO_TOKEN_TYPE[type_index]
        for type_index in batch['token_type_index'][index_in_batch].numpy()[keep]
    ]
    delta_time = batch['delta_time'][index_in_batch].numpy()[keep]

    magnitude_mask = batch['magnitude_mask'][index_in_batch].numpy()[keep].astype(bool)
    sigma_mask = batch['sigma_mask'][index_in_batch].numpy()[keep].astype(bool)
    mag_model = batch['magnitude'][index_in_batch].numpy()[keep] * MAG_STD + MAG_MEAN
    sigma_model = batch['sigma_magnitude'][index_in_batch].numpy()[keep] * SIGMA_MAG_STD + SIGMA_MAG_MEAN
    mag_model = np.where(magnitude_mask, mag_model, np.nan)
    sigma_model = np.where(sigma_mask, sigma_model, np.nan)

    flux_true = batch['flux_true'][index_in_batch].numpy()[keep]
    fluxcal_err = batch['fluxcal_err'][index_in_batch].numpy()[keep]
    zp = batch['zp'][index_in_batch].numpy()[keep]

    mag_true = ZP_FLUXCAL - 2.5 * np.log10(flux_true)
    maglim_5sig = ZP_FLUXCAL - 2.5 * np.log10(SNR_MIN * fluxcal_err)
    alpha = 10.0 ** ((ZP_FLUXCAL - zp) / 2.5)

    return pd.DataFrame({
        'band': band,
        'type': token_type,
        'dt': delta_time,
        'mag_model': mag_model,
        'sigma_model': sigma_model,
        'mag_true': mag_true,
        'maglim_5sig': maglim_5sig,
        'flux_true': flux_true,
        'fluxcal_err': fluxcal_err,
        'zp': zp,
        'alpha': alpha,
    })


human_view = decode_batch_object_to_magnitudes(example_batch, index_in_batch=0)
print(human_view.to_string(index=False, float_format=lambda value: f'{value:8.3f}'))
human_view


In [123]:
# ## Batch-level scenario table — the observational template handed to the KN dataloader
#
# Flatten the whole collated batch into one long table (one row per real, non-padded token of
# every object). The KN dataloader builds each twin on this exact (object, dt, band,
# token_type, zp) grid, so it inherits every contaminant's REALIZED epochs — window shift and
# 1/2/3-epoch prefix truncation already applied — instead of re-sampling them. This is what
# couples the two dataloaders: same observational scenario per batch by construction.

def batch_scenario_table(batch):
    """Long per-token table for the whole batch, to feed the KN dataloader.

    Per object the KN twin reuses (dt, band, token_type) and the scenario (flux_true,
    fluxcal_err, zp) to set its matched error
        sigma_KN^2 = fluxcal_err^2 + alpha * (F_KN - flux_true),   alpha = 10^((27.5 - zp)/2.5)
    and places its SED at true_redshift. The mag_* columns are the human-magnitude
    cross-checks from decode_batch_object_to_magnitudes (model vs noiseless vs 5 sigma depth).
    'n' tokens are kept (NaN scenario) so the twin reproduces the same token layout.
    """
    object_tables = []
    for index_in_batch in range(batch['delta_time'].shape[0]):
        object_table = decode_batch_object_to_magnitudes(batch, index_in_batch)
        object_table.insert(0, 'object', index_in_batch)
        object_table.insert(1, 'cid', int(batch['cid'][index_in_batch]))
        object_table.insert(2, 'label', GROUP_ORDER[int(batch['label'][index_in_batch])])
        object_table.insert(3, 'true_redshift', float(batch['true_redshift'][index_in_batch]))
        object_tables.append(object_table)
    return pd.concat(object_tables, ignore_index=True)


scenario_table = batch_scenario_table(example_batch)
print(f'tokens: {len(scenario_table):,}   objects: {scenario_table["object"].nunique()}')
print(scenario_table.head(16).to_string(index=False, float_format=lambda value: f'{value:8.3f}'))
scenario_table


tokens: 532   objects: 64
 object    cid label  true_redshift band type       dt  mag_model  sigma_model  mag_true  maglim_5sig  flux_true  fluxcal_err       zp    alpha
      0  32920    Ia          0.903    R    d    0.000     26.189        0.143    26.287       26.645      3.055        0.440   32.129    0.014
      0  32920    Ia          0.903    Z    d    0.000     25.312        0.152    25.374       25.696      7.083        1.053   31.303    0.030
      0  32920    Ia          0.903    Y    d    0.000     24.934        0.116    25.427       25.619      6.745        1.131   31.356    0.029
      0  32920    Ia          0.903    J    d    0.000     25.082        0.139    25.517       25.564      6.213        1.189   31.354    0.029
      0  32920    Ia          0.903    R    d    5.000     26.159        0.139    25.595       26.641      5.780        0.441   32.129    0.014
      0  32920    Ia          0.903    Z    d    5.000     24.909        0.110    24.738       25.649     12.7

,object,cid,label,true_redshift,band,type,dt,mag_model,sigma_model,mag_true,maglim_5sig,flux_true,fluxcal_err,zp,alpha
0,0,32920,Ia,0.903151,R,d,0.0,26.189314,0.142746,26.287489,26.644789,3.054952,0.439658,32.129002,0.014073
1,0,32920,Ia,0.903151,Z,d,0.0,25.311682,0.152378,25.374420,25.696259,7.083240,1.053238,31.302999,0.030116
2,0,32920,Ia,0.903151,Y,d,0.0,24.933928,0.115575,25.427475,25.618652,6.745436,1.131277,31.356001,0.028681
3,0,32920,Ia,0.903151,J,d,0.0,25.081886,0.139242,25.516809,25.564348,6.212640,1.189299,31.354000,0.028734
4,0,32920,Ia,0.903151,R,d,5.0,26.158630,0.139252,25.595148,26.641012,5.780172,0.441189,32.129002,0.014073
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
527,63,164619,II,1.679270,F,u,5.0,26.821087,NaN,26.567932,26.821087,2.359539,0.373762,33.299000,0.004791
528,63,164619,II,1.679270,Y,u,10.0,26.887573,NaN,28.487940,26.887573,0.402554,0.351561,32.549000,0.009559
529,63,164619,II,1.679270,J,u,10.0,26.780449,NaN,27.269409,26.780449,1.236620,0.388017,32.547001,0.009576
530,63,164619,II,1.679270,H,u,10.0,26.669701,NaN,26.811428,26.669701,1.885510,0.429684,32.570000,0.009376


In [128]:
# ## Bright low-z batch — a controlled scenario template to exercise the KN dataloader (part 2)
#
# A random survey batch is mostly z > 0.5, where an injected KN twin sits far below the 5σ
# depth — useless for checking part 2. This builds an *artificial* batch from only the
# brightest, lowest-redshift contaminants (z < max_redshift), so every object's matched twin
# is built at a redshift where a KN is actually detectable (the LANL grid peaks ~M_AB -13,
# detectable only z ≲ 0.15). data_aug=False: full 3-epoch window and true z — the clean
# scenario to validate part 2 against. Feeds straight into batch_scenario_table.

def build_low_z_bright_batch(max_redshift=0.2, n_objects=512, data_aug=False,
                             random_seed=BATCH_SEED):
    """Collate a batch of the brightest contaminants below max_redshift. Brightness is the
    object's peak detection (smallest 'd'-token magnitude). n_objects=None keeps all of them.
    Returns (collated_batch, summary_table).
    """
    peak_mag_by_cid = phot[phot['token_type'] == 'd'].groupby('cid')['mag'].min()

    candidate_cids = [
        cid for cid in kept_cids
        if redshift_by_cid[cid] < max_redshift and cid in peak_mag_by_cid.index
    ]
    candidate_cids.sort(key=lambda cid: peak_mag_by_cid[cid])  # brightest (smallest mag) first
    if n_objects is not None:
        candidate_cids = candidate_cids[:n_objects]

    dataset = HourglassWindowDataset(
        candidate_cids, photometry_by_cid, redshift_by_cid, label_by_cid,
        data_aug=data_aug, random_seed=random_seed,
    )
    batch = collate_token_windows([dataset[index] for index in range(len(dataset))])

    summary = pd.DataFrame({
        'cid': candidate_cids,
        'label': [GROUP_ORDER[label_by_cid[cid]] for cid in candidate_cids],
        'z_cmb': [redshift_by_cid[cid] for cid in candidate_cids],
        'peak_mag': [float(peak_mag_by_cid[cid]) for cid in candidate_cids],
    })
    return batch, summary


low_z_bright_batch, low_z_bright_summary = build_low_z_bright_batch(max_redshift=0.3)
print(
    f'objects in batch: {len(low_z_bright_summary)}   '
    f'z in [{low_z_bright_summary["z_cmb"].min():.4f}, {low_z_bright_summary["z_cmb"].max():.4f}]   '
    f'peak mag in [{low_z_bright_summary["peak_mag"].min():.2f}, {low_z_bright_summary["peak_mag"].max():.2f}]'
)
print(low_z_bright_summary['label'].value_counts().to_string())

# the observational template the KN dataloader (part 2) consumes
low_z_bright_scenario = batch_scenario_table(low_z_bright_batch)
print(f'scenario tokens: {len(low_z_bright_scenario):,}   objects: {low_z_bright_scenario["object"].nunique()}')
low_z_bright_scenario.to_csv('example_1_batch_token_brillante.csv', index=False)
low_z_bright_summary


objects in batch: 512   z in [0.0371, 0.2997]   peak mag in [17.64, 22.23]
label
II       246
Ia       236
other     30
scenario tokens: 6,136   objects: 512


,cid,label,z_cmb,peak_mag
0,10788,Ia,0.053883,17.636124
1,175037,II,0.041002,18.302528
2,24520,Ia,0.065667,18.604160
3,96161,II,0.037123,18.758608
4,125877,II,0.039449,18.865967
...,...,...,...,...
507,141523,II,0.175691,22.225874
508,210944,II,0.248180,22.225941
509,142321,II,0.260818,22.230152
510,55863,II,0.276120,22.230326
